#### Parallel Chains

Based on the text, model will run parallelly and gives the summary of the text and one quiz from the text.

In [4]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableSequence, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

In [2]:
load_dotenv()

True

In [5]:
# Model
model = ChatOpenAI(model='gpt-4.1-mini', max_completion_tokens=200)

In [7]:
# String Parser
parser = StrOutputParser()

In [9]:
# Summary prompt
summary_prompt = PromptTemplate(
    template='Give me the summary of the text: {txt}',
    input_variables={'txt'}
)

# quiz prompt
quiz_prompt = PromptTemplate(
    template='Give me one quiz from the text: {txt}',
    input_variables={'txt'}
)

In [10]:
# Summary chain
summary_chain = summary_prompt | model | parser


# quiz chain
quiz_chain = quiz_prompt | model | parser

# Parallel chain
combined_chain = RunnableParallel(
    {
        "summary": summary_chain,
        "quiz": quiz_chain
    }
)

In [11]:
res = combined_chain.invoke({
    "txt": """Fine-tuning: We fine-tune the model using a contrastive objective. Formally, we compute the cosine similarity from each possible sentence pairs from the batch. We then apply the cross entropy loss by comparing with true pairs."""
})

res

{'summary': 'The text describes a fine-tuning process where the model is trained using a contrastive objective. It involves calculating the cosine similarity between all sentence pairs in a batch and then applying cross-entropy loss by comparing these similarities with the true sentence pairs.',
 'quiz': "Sure! Here's a quiz question based on the text you provided:\n\n**Quiz Question:**  \nWhat type of loss function is used when fine-tuning the model with a contrastive objective involving cosine similarity between sentence pairs?  \n\nA) Mean Squared Error  \nB) Cross Entropy Loss  \nC) Hinge Loss  \nD) KL Divergence  \n\n**Answer:**  \nB) Cross Entropy Loss"}

In [12]:
print(res['summary'])

The text describes a fine-tuning process where the model is trained using a contrastive objective. It involves calculating the cosine similarity between all sentence pairs in a batch and then applying cross-entropy loss by comparing these similarities with the true sentence pairs.


In [13]:
print(res['quiz'])

Sure! Here's a quiz question based on the text you provided:

**Quiz Question:**  
What type of loss function is used when fine-tuning the model with a contrastive objective involving cosine similarity between sentence pairs?  

A) Mean Squared Error  
B) Cross Entropy Loss  
C) Hinge Loss  
D) KL Divergence  

**Answer:**  
B) Cross Entropy Loss


In [16]:
# Chain Graph
combined_chain.get_graph().print_ascii()

          +-----------------------------+            
          | Parallel<summary,quiz>Input |            
          +-----------------------------+            
                 **               **                 
              ***                   ***              
            **                         **            
+----------------+                +----------------+ 
| PromptTemplate |                | PromptTemplate | 
+----------------+                +----------------+ 
          *                               *          
          *                               *          
          *                               *          
  +------------+                    +------------+   
  | ChatOpenAI |                    | ChatOpenAI |   
  +------------+                    +------------+   
          *                               *          
          *                               *          
          *                               *          
+-----------------+         